# RD03 — Regime-dependent returns against AM-T-HAR-Blend

**Primary:** 30-minute conditional signed-return density. **Secondary:** 5, 15, 60-minute density. **Frozen baseline:** actual `Volatility_Dynamics/18/18_manifest.json` parameters, not an RV proxy. **Status:** historical exploratory, *not* independent confirmation.

Seven matched methods: unchanged AM-T-HAR-Blend, sign-only return mean, continuous return mean, observable CART return-mean state, static HMM emission state, causal HMM state, and age-dependent semi-Markov overlay. Each method uses identical archived VD18 variance particles and HAR calibration. No model refit uses Sep–Oct 2026 evaluation labels.

Read `discussions/Regime_Discovery/003_chatgpt_regime_density_bridge_preregistration.md` first. `RD03_SYNTHETIC=1` tests only arithmetic/output mechanics, not numerical VD18 archive reconstruction.

In [ ]:
import sys,os,json,time,subprocess,tempfile,gc
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(),'Run inside repository checkout'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import regime_density_bridge as rd
cfg=rd.Specification()
SYNTHETIC=os.getenv('RD03_SYNTHETIC','0')=='1'
if SYNTHETIC:
    TEMP=tempfile.TemporaryDirectory(prefix='rd03-')
    OUT=Path(TEMP.name)/'03'
else:
    data_root=os.getenv('DERIBIT_DATA_ROOT')
    if not data_root:raise RuntimeError('DERIBIT_DATA_ROOT missing')
    STORE=Path(data_root)/'research'/'cs04a_feature_store'
    HIST=Path(data_root)/'history'/'price_5m'
    ARCHIVE=ROOT/'notebooks'/'Volatility_Dynamics'/'18'/'18_manifest.json'
    OUT=ROOT/'notebooks'/'Regime_Discovery'/'03'
OUT.mkdir(parents=True,exist_ok=True)
print(rd.VERSION,'synthetic=',SYNTHETIC,'output:',OUT)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
tests=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_regime_density_bridge.py')],
  cwd=str(ROOT),env=env,capture_output=True,text=True)
print(tests.stdout[-1800:]);print(tests.stderr[-700:]);assert tests.returncode==0,'RD03 regressions failed'


In [ ]:
if SYNTHETIC:
    rng=np.random.default_rng(cfg.seed)
    fake=[]
    for asset in cfg.assets:
        for h in cfg.horizons:
            for day in pd.date_range('2026-09-19',periods=18,freq='D',tz='UTC'):
                v=rng.uniform(1e-7,1e-5,(48,16));y=rng.normal(0,.001,48)
                for model in rd.METHODS:
                    mu=np.zeros(48) if model=='AM_T_HAR_BLEND' else rng.normal(0,.00002,48)
                    z=rd.normal_mixture_scores(y,v,mu)
                    fake.append({'asset':asset,'horizon_min':h,'method':model,'date':day,
                                 'nll':float(z['nll'].mean()),'up_brier':float(z['up_brier'].mean())})
    aggregate=pd.DataFrame(fake)
    scores=aggregate.groupby(['asset','horizon_min','method'],as_index=False).agg(
          nll=('nll','mean'),up_brier=('up_brier','mean'),n_days=('date','nunique'))
    boots=[]
    for asset,g in aggregate.groupby('asset'):
        boots.append(rd.paired_day_bootstrap(g,cfg).assign(asset=asset))
    bootstrap=pd.concat(boots,ignore_index=True)
    diagnostics=pd.DataFrame([{'asset':asset,'synthetic':True} for asset in cfg.assets])
    split_audit={'synthetic':True}
    excluded={}
else:
    manifest=json.loads(ARCHIVE.read_text(encoding='utf-8'))
    params=rd.confirm_parameters(manifest,cfg)
    print('Frozen VD18 archive verified:',manifest['registration']['version'],
          'last market observation:',manifest['sample_last'])
    vdframe,split_audit=rd.archived_split_frame(HIST,cfg)
    print('Archived split reconstruction:',split_audit)
    print('Historical VD18 rows:',len(vdframe))
    from data_engineering.modeling import markov_conditional_variance as v17
    scores_list=[];boot_list=[];diagnostic_list=[];excluded={}
    for asset in cfg.assets:
        start=time.monotonic()
        path=STORE/f'{asset}_features_and_forward_magnitude.parquet'
        if not path.exists():excluded[asset]=f'local CS04-A feature store missing: {path}';continue
        required=list(dict.fromkeys(['state_time','feature_ready',*rd.STATE_FEATURES,*rd.RETURN_FEATURES]))
        data=pd.read_parquet(path,columns=required)
        observed=rd.merge_market_features(vdframe,data,asset)
        observed=rd.periods(observed,cfg)
        # RD03 holds out the same evaluation origins for every method and horizon.
        vars_by_h={}
        for h in cfg.horizons:
            vars_by_h[h]=rd.vd18_particle_variances(observed,asset,h,params,
                    particles=cfg.particle_count,block=256)
            assert len(vars_by_h[h])==len(observed)
            # Exact unchanged density NLL identity against existing VD17 implementation.
            check=observed[f'R_{h}'].to_numpy(float)[:64]
            ref=v17.loss_components(vars_by_h[h][:64],check,np.full(len(check),.001))['nll']
            ours=rd.normal_mixture_scores(check,vars_by_h[h][:64],np.zeros(len(check)))['nll']
            assert np.allclose(ref,ours,rtol=1e-11,atol=1e-11),f'VD18 {asset} {h} identity failure'
        sc,origin_losses,diag=rd.score_asset(observed,vars_by_h,cfg)
        scores_list.append(sc.assign(asset=asset))
        boot_list.append(rd.paired_day_bootstrap(origin_losses,cfg).assign(asset=asset))
        diagnostic_list.append({'asset':asset,'n_structure':diag['n_structure'],
                'n_response':diag['n_response'],'n_evaluation':diag['n_evaluation'],
                'tree_rules':diag['tree'],'hmm_transition_matrix':json.dumps(diag['hmm_transitions']),
                'age_hazard':json.dumps(diag['hazards']),
                'response_parameters':json.dumps(diag['horizon_response'])})
        print(asset,'evaluated',diag['n_evaluation'],'anchor origins in',round(time.monotonic()-start,1),'seconds')
        del observed,vars_by_h,data,sc,origin_losses,diag
        gc.collect()
    if not {'BTC','ETH'}.issubset(set(cfg.assets)-set(excluded)):
        raise RuntimeError(f'Primary coverage incomplete: {excluded}')
    scores=pd.concat(scores_list,ignore_index=True)
    bootstrap=pd.concat(boot_list,ignore_index=True)
    diagnostics=pd.DataFrame(diagnostic_list)
print('Evaluated models:',len(scores),'asset×horizon×method cells')
display(scores.query('horizon_min==30').sort_values(['asset','nll']).head(24))


In [ ]:
# Descriptive comparisons to the ORIGINAL VD18 density and to direct feature conditioning.
reference=scores[scores.method.eq('AM_T_HAR_BLEND')].set_index(['asset','horizon_min'])
direct=scores[scores.method.eq('direct_mean')].set_index(['asset','horizon_min'])
rows=[]
for row in scores.itertuples(index=False):
    key=(row.asset,row.horizon_min)
    rows.append({'asset':row.asset,'horizon_min':row.horizon_min,'method':row.method,
       'nll':row.nll,'delta_nll_vs_vd18':row.nll-reference.loc[key,'nll'],
       'delta_nll_vs_direct':row.nll-direct.loc[key,'nll'],
       'up_brier':row.up_brier,'delta_up_brier_vs_vd18':row.up_brier-reference.loc[key,'up_brier']})
comparison=pd.DataFrame(rows)
print('30m mean delta signed-return log density, negative better:')
display(comparison.query('horizon_min==30').sort_values(['asset','delta_nll_vs_vd18']))
print('Exploratory paired-day intervals; not independent confirmation:')
display(bootstrap.query('horizon_min==30 and metric=="nll"').sort_values(['asset','delta_equal_day']))
verdict=rd.decision(scores,bootstrap,cfg,synthetic=SYNTHETIC)
print(json.dumps(verdict,indent=2))


In [ ]:
# Commit only SMALL AGGREGATE results. Synthetic output goes to a temporary folder.
mapping={'03_scores.csv':scores,'03_paired_day_bootstrap.csv':bootstrap,
         '03_comparison_summary.csv':comparison,'03_model_diagnostics.csv':diagnostics}
for name,df in mapping.items():df.to_csv(OUT/name,index=False)
meta={'module':'RD03','registration':rd.registration(),
      'baseline':'VD18 archived AM_T_HAR_BLEND frozen parameters, not RV proxy',
      'source_archive':'notebooks/Volatility_Dynamics/18/18_manifest.json',
      'synthetic':SYNTHETIC,'assets_evaluated':sorted(scores.asset.unique()),
      'excluded':excluded,'archive_split_audit':split_audit,
      'horizons':list(cfg.horizons),'methods':list(rd.METHODS),
      'status':cfg.status if not SYNTHETIC else 'SYNTHETIC_NO_MARKET_INFERENCE',
      'historical_period_previously_inspected':True,
      'variance_particles_identical_across_methods':True,
      'pointwise_vd17_nll_identity_tested':not SYNTHETIC,
      'AM_T_HAR_Blend_modified':False}
(OUT/'03_manifest.json').write_text(json.dumps(meta,indent=2,default=str),encoding='utf-8')
(OUT/'03_decision.json').write_text(json.dumps(verdict,indent=2),encoding='utf-8')
assert all((OUT/f).is_file() for f in list(mapping)+['03_manifest.json','03_decision.json'])
print('Saved',len(mapping)+2,'aggregate files to',OUT)
print('DO NOT push per-origin data, variance particles, filtered posteriors or synthetic output.')


## Interpretation and restrictions

- **Do not call this a prospective result.** September–October 2026 dates were already inspected in CS03, CS04, and other branches.
- `latent_semi_markov_mean` uses RD02's two-stage age-hazard approximation; it is not a joint HSMM.
- HMM current-state probabilities are causally filtered from completed features; **Viterbi is never used at prediction time**.
- The state response is a single filtered *location* shift of VD18's original Student-t variance mixture. This experiment does not forecast state-dependent skew, variance, or mixture multimodality.
- **No modifications to VD18, AM-T-HAR-Blend, Deribit collectors or options execution.**
